# Práctica 6 - Preprocesamiento de datos para Machine Learning
**Sistemas de Aprendizaje Automático · Sesión 6**   
**Dataset:** `clientes_ecommerce.csv` · **Modalidad:** individual

## Contexto
Trabajas en una empresa de comercio electrónico que quiere predecir si un cliente comprará el próximo mes. El dataset ya se exploró en las sesiones 4 y 5. Ahora debes transformarlo para que esté listo para entrenar un modelo de clasificación binaria.

**No entrenes modelos en esta práctica.** Completa los bloques `TODO` y responde las preguntas en las celdas Markdown de respuesta.

### Objetivos
- Detectar ausentes y anomalías
- Elegir estrategias de imputación 
- Codificar variables nominales y ordinales
- Comparar escalados
- Separar entrenamiento/prueba
- Construir un `Pipeline` con `ColumnTransformer` 
- Evitar data leakage
- Verificar las matrices resultantes

> Coloca `clientes_ecommerce.csv` en la misma carpeta que este notebook. Usa scikit-learn 1.2 o posterior para `sparse_output=False`.

## Parte 1 - Diagnóstico del dataset
### Ejercicio 1. Carga y estructura
Carga el CSV, muestra 10 filas, dimensiones, tipos de datos, información general y número de registros duplicados.

In [1]:
# scikit-learn ya debería estar instalado.
# Si falla el import más abajo, ejecuta SOLO esta línea una vez:
# %pip install scikit-learn

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Cargar el CSV
df = pd.read_csv('clientes_ecommerce.csv')

# 10 primeras filas
print(df.head(10))

# Dimensiones, tipos, info y duplicados
print('Shape:', df.shape)
print(df.dtypes)
print(df.info())
print('Duplicados:', df.duplicated().sum())


   cliente_id  edad  antiguedad_meses  ingresos_anuales  gasto_mensual  \
0       20001    67             102.0           9000.00          60.73   
1       20002    28              93.0          35683.86         131.74   
2       20003    19              87.0          37152.19          55.33   
3       20004    55              79.0          40158.29          24.33   
4       20005    39               3.0          27317.06          88.78   
5       20006    45              25.0          32627.01          59.77   
6       20007    22             105.0          44101.68         109.67   
7       20008    39              90.0          44882.10          57.56   
8       20009    55              56.0          55299.19          49.94   
9       20010    38              93.0           9000.00         110.69   

   num_compras  devoluciones dispositivo    zona nivel_fidelidad  \
0            7             3       Móvil   Oeste             Oro   
1            6             1       Móvil    Este 

**Responde:** ¿Cuántas muestras y columnas hay? ¿Cuál es el target? ¿Qué columna identifica a cada cliente? ¿Hay filas duplicadas?

**Respuesta:**

- Hay **400 muestras** y **12 columnas**.
- El **target** es `compra_proximo_mes` (Sí / No).
- El identificador de cada cliente es `cliente_id`.
- **No hay filas duplicadas** (0 duplicados).


### Ejercicio 2. Valores ausentes
Calcula el número de ausentes por columna, el porcentaje y el número de filas con al menos un ausente. Completa la tabla.

In [3]:
# Ausentes por columna
ausentes = df.isna().sum()
print(ausentes)

# Porcentaje de ausentes por columna
porcentaje = df.isna().mean() * 100
print(porcentaje)

# Filas con algún valor ausente
filas_incompletas = df.isna().any(axis=1).sum()
print('Filas incompletas:', filas_incompletas)


cliente_id            0
edad                  0
antiguedad_meses      5
ingresos_anuales      4
gasto_mensual         3
num_compras           0
devoluciones          0
dispositivo           0
zona                  0
nivel_fidelidad       0
satisfaccion          0
compra_proximo_mes    0
dtype: int64
cliente_id            0.00
edad                  0.00
antiguedad_meses      1.25
ingresos_anuales      1.00
gasto_mensual         0.75
num_compras           0.00
devoluciones          0.00
dispositivo           0.00
zona                  0.00
nivel_fidelidad       0.00
satisfaccion          0.00
compra_proximo_mes    0.00
dtype: float64
Filas incompletas: 12


| Variable | Nº ausentes | Porcentaje |
|---|---:|---:|
| `antiguedad_meses` | 5 | 1.25 % |
| `ingresos_anuales` | 4 | 1.00 % |
| `gasto_mensual` | 3 | 0.75 % |

Hay **12 filas** con al menos un ausente.

**¿Eliminarías todas las filas incompletas? ¿Por qué?**

**Respuesta:**  
No. Son pocas filas, pero al borrarlas perdemos información. Es mejor **imputar** (mediana/moda) y conservar los datos.


### Ejercicio 3. Edad sospechosa
Localiza edades superiores a 120 años, muestra `cliente_id` y `edad` y consulta mínimo y máximo.

In [4]:
# Mínimo y máximo de edad
print('Edad mínima:', df['edad'].min())
print('Edad máxima:', df['edad'].max())

# Registros con edad > 120
print(df.loc[df['edad'] > 120, ['cliente_id', 'edad']])


Edad mínima: 18
Edad máxima: 195
     cliente_id  edad
137       20138   195


**Responde:** ¿Qué harías con esa edad? ¿Por qué no deberías cambiar 195 por 95 sin evidencia?

**Respuesta:**  
Marcaría 195 como valor inválido (por ejemplo convertirlo a `NaN`) y luego imputarlo.  
No cambiaría 195 por 95 sin evidencia porque **no sabemos** si fue un error de tecleo: podría ser otro fallo distinto.


## Parte 2 - Variables y división
### Ejercicio 4. Clasificación de columnas
Clasifica las columnas según su naturaleza.

| Tipo | Columnas |
|---|---|
| Numéricas | `edad`, `antiguedad_meses`, `ingresos_anuales`, `gasto_mensual`, `num_compras`, `devoluciones`, `satisfaccion` |
| Categóricas nominales | `dispositivo`, `zona` |
| Categóricas ordinales | `nivel_fidelidad` |
| Identificador | `cliente_id` |
| Target | `compra_proximo_mes` |

**Responde:** ¿Por qué `zona` es nominal y `nivel_fidelidad` ordinal? ¿Por qué no incluirías `cliente_id` en el modelo?

**Respuesta:**  
- `zona` es **nominal** porque Norte/Sur/Este/Oeste no tienen un orden natural.  
- `nivel_fidelidad` es **ordinal** porque Bronce < Plata < Oro < Platino sí tiene orden.  
- `cliente_id` no se usa en el modelo porque solo identifica al cliente y **no aporta información predictiva**.


### Ejercicio 5. Separar X e y
Excluye de X el identificador y el target. Codifica `No` como 0 y `Sí` como 1. Comprueba dimensiones y distribución de y.

In [5]:
# TODO: X = ... (sin cliente_id ni compra_proximo_mes)

X = df.drop(columns=['cliente_id', 'compra_proximo_mes'])
y = df['compra_proximo_mes'].map({'No':0, 'Sí':1})

# TODO: X.shape, y.shape, y.value_counts()

print(X.shape)
print(y.shape)
print(y.value_counts())

(400, 10)
(400,)
compra_proximo_mes
0    246
1    154
Name: count, dtype: int64


**¿Cuántas características predictoras iniciales hay?**

**Respuesta:**  
Hay **10** características predictoras iniciales (todas las columnas de `X`).


### Ejercicio 6. División entrenamiento/prueba
Separa 80 % para entrenamiento y 20 % para prueba. Utiliza `random_state=42` y `stratify=y`. Comprueba tamaños.

In [6]:
# TODO: completar

X_train, X_test, y_train, y_test = train_test_split(

X, y, test_size = 0.20, random_state = 42, stratify = y)

# TODO: mostrar las dimensiones de los cuatro conjuntos

print('Entrenamiento: ',X_train.shape, y_train.shape)
print('Test: ',X_test.shape, y_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))



Entrenamiento:  (320, 10) (320,)
Test:  (80, 10) (80,)
compra_proximo_mes
0    0.615625
1    0.384375
Name: proportion, dtype: float64
compra_proximo_mes
0    0.6125
1    0.3875
Name: proportion, dtype: float64


**Responde:** ¿Cuántas muestras hay en cada conjunto? ¿Para qué sirve `stratify=y`? ¿Por qué dividimos antes de ajustar transformaciones?

**Respuesta:**  
- Entrenamiento: **320** muestras. Prueba: **80** muestras.  
- `stratify=y` mantiene en train y test la **misma proporción** de Sí/No que en el dataset original.  
- Dividimos antes para que las transformaciones se aprendan **solo con entrenamiento** y no haya *data leakage* (filtración de información de prueba).


## Parte 3 - Imputación y encoding
### Ejercicio 7. Validación de edad
Sobre **copias** de `X_train` y `X_test`, sustituye edades superiores a 120 por `np.nan`. No imputes todavía: eso se hará en el pipeline. El umbral es una regla didáctica de validación del dominio.

In [7]:
# IMPUTACIÓN se utiliza cuando un valor es NaN, y susitituimos ese valor vacío por un valor como es la MEDIA,MEDIANA,MODA Y CONSTANTE
# CODIFICACIÓN es para la clasificacion de datos mediante ONE HOP ENCODING O COD. ORDINAL 
# ESCALADO se utiliza para escalar los datos entre unos valores, en MAX/MIN(ENTRE 0 Y 1) O EN EL Z-SCORE(MEDIA Y DESVIACION TIP)

In [8]:
X_train = X_train.copy()
X_test = X_test.copy()

# Marcar edades > 120 como NaN (regla de validación del dominio)
X_train.loc[X_train['edad'] > 120, 'edad'] = np.nan
X_test.loc[X_test['edad'] > 120, 'edad'] = np.nan

# Contar ausentes en edad
print('Ausentes edad (train):', X_train['edad'].isna().sum())
print('Ausentes edad (test):', X_test['edad'].isna().sum())


Ausentes edad (train): 1
Ausentes edad (test): 0


### Ejercicio 8. Estrategias de imputación
Utilizaremos mediana para numéricas y moda para categóricas. Crea ambos objetos `SimpleImputer` **sin ajustarlos todavía**.

In [9]:
"""
Strategy:
-Media = mean
-Mediana = median
-Moda = most_frequent
-valor contante : contant=> fill_value=x

"""
#TODO: imputer_numerico = SimpleImputer(strategy="constant"; fill_value = 20)

imputer_numerico = SimpleImputer(strategy="median")
#TODO: imputer_categorico = SimpleImputer(...)

imputer_categorico = SimpleImputer(strategy='most_frequent')


| Variable | Estrategia | Justificación |
|---|---|---|
| edad | mediana | Es numérica y la mediana aguanta mejor los valores raros |
| antiguedad_meses | mediana | Numérica con pocos ausentes |
| ingresos_anuales | mediana | Numérica; puede tener valores extremos |
| gasto_mensual | mediana | Numérica; misma idea |
| dispositivo | moda | Categórica: rellenamos con la categoría más frecuente |
| zona | moda | Categórica: misma idea |

**¿Por qué no ajustar el imputador con entrenamiento y prueba a la vez?**

**Respuesta:**  
Porque el imputador debe aprender la mediana/moda **solo del entrenamiento**. Si usa también la prueba, se filtra información y la evaluación queda sesgada.


### Ejercicio 9. One-Hot Encoding
Prueba `OneHotEncoder` con un DataFrame independiente. Configura `handle_unknown="ignore"` y `sparse_output=False`. Muestra matriz y nombres de columnas.

In [10]:
from json import encoder


ejemplo = pd.DataFrame({
    "dispositivo": ["Móvil", "Ordenador", "Tablet", "Móvil"]
})

# TODO: encoder_nominal = OneHotEncoder(...)

encoder_nominal = OneHotEncoder(handle_unknown = 'ignore', sparse_output = False)

# TODO: aplicar fit_transform a ejemplo

matriz_ejemplo = encoder_nominal.fit_transform(ejemplo)

# TODO: consultar get_feature_names_out()

print(matriz_ejemplo)
pd.DataFrame(matriz_ejemplo, columns = encoder_nominal.get_feature_names_out())

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]


,dispositivo_Móvil,dispositivo_Ordenador,dispositivo_Tablet
0,1.0,0.0,0.0
1,0.0,1.0,0.0
2,0.0,0.0,1.0
3,1.0,0.0,0.0


**Responde:** ¿Cuántas columnas se generan? ¿Qué representan? ¿Por qué no codificar Móvil=0, Ordenador=1 y Tablet=2 como categorías ordenadas?

**Respuesta:**  
- Se generan **3 columnas** (una por categoría).  
- Cada columna indica con 0/1 si esa fila es de esa categoría.  
- No conviene poner 0, 1, 2 porque el modelo pensaría que hay un **orden** (Tablet > Ordenador > Móvil) y aquí no lo hay.


### Ejercicio 10. Codificación ordinal
Configura `OrdinalEncoder` para el orden Bronce, Plata, Oro, Platino. Comprueba su funcionamiento con un ejemplo independiente.

In [11]:
ejemplo_fidelidad = pd.DataFrame({
    "nivel_fidelidad": ["Oro", "Bronce", "Platino", "Plata"]
})

# TODO: encoder_ordinal = OrdinalEncoder(categories=[[...]])
encoder_ordinal = OrdinalEncoder(categories=[['Bronce', 'Plata', 'Oro', 'Platino']])

# TODO: transformar ejemplo_fidelidad y mostrar el resultado
resultado = encoder_ordinal.fit_transform(ejemplo_fidelidad)
print(resultado)
print(resultado.ravel())

pd.DataFrame({'nivel':ejemplo_fidelidad['nivel_fidelidad'],'codigo':resultado.ravel()})

[[2.]
 [0.]
 [3.]
 [1.]]
[2. 0. 3. 1.]


,nivel,codigo
0,Oro,2.0
1,Bronce,0.0
2,Platino,3.0
3,Plata,1.0


**Responde:** ¿Qué representan los códigos? ¿La distancia entre niveles consecutivos es necesariamente igual?

**Respuesta:**  
Los códigos representan el **orden** del nivel de fidelidad: Bronce=0, Plata=1, Oro=2, Platino=3.  
La distancia entre niveles **no tiene por qué ser igual** en la realidad; el encoder solo guarda el orden.


## Parte 4 - Escalado
### Ejercicio 11. Min-Max frente a Z-score
Aplica ambos escaladores al siguiente ejemplo. Redondea a tres decimales y completa la tabla.

In [12]:
datos = pd.DataFrame({"valor": [10, 20, 30, 40, 50]})

# Crear los dos escaladores
minmax = MinMaxScaler()
zscore = StandardScaler()

# Transformar
datos_minmax = minmax.fit_transform(datos)
datos_zscore = zscore.fit_transform(datos)

# Tabla comparativa
comparacion = pd.DataFrame({
    'Original': datos['valor'],
    'Min-Max': np.round(datos_minmax.ravel(), 3),
    'StandardScaler': np.round(datos_zscore.ravel(), 3)
})
print(comparacion)


   Original  Min-Max  StandardScaler
0        10     0.00          -1.414
1        20     0.25          -0.707
2        30     0.50           0.000
3        40     0.75           0.707
4        50     1.00           1.414


| Original | Min-Max | StandardScaler |
|---:|---:|---:|
| 10 | 0.000 | -1.414 |
| 20 | 0.250 | -0.707 |
| 30 | 0.500 | 0.000 |
| 40 | 0.750 | 0.707 |
| 50 | 1.000 | 1.414 |

**Responde:** ¿Qué rango da Min-Max en este ejemplo? ¿Qué valor toma la media tras estandarizar? ¿Pueden aparecer valores negativos? ¿Qué escalado elegirías para un algoritmo basado en distancias y por qué?

**Respuesta:**  
- Min-Max deja los valores entre **0 y 1**.  
- Tras estandarizar, la media queda en **0**.  
- Sí: con StandardScaler pueden salir **valores negativos**.  
- Para algoritmos por distancias (por ejemplo KNN) suele ir bien **StandardScaler** o Min-Max, porque así ninguna variable domina solo por tener números más grandes. En esta práctica usaremos estandarización en el pipeline numérico.


### Ejercicio 12. Variables que escalaremos
Para esta práctica trataremos `satisfaccion` como numérica, aunque conceptualmente es una escala ordinal de 1 a 5. Es una simplificación que deberá revisarse en otros modelos.

In [13]:
numericas = [
    "edad", "antiguedad_meses", "ingresos_anuales",
    "gasto_mensual", "num_compras", "devoluciones", "satisfaccion"
]
nominales = ["dispositivo", "zona"]
ordinales = ["nivel_fidelidad"]

# Comprobar que las columnas existen en X_train
print('¿Están las numéricas?', all(col in X_train.columns for col in numericas))
print('¿Están las nominales?', all(col in X_train.columns for col in nominales))
print('¿Están las ordinales?', all(col in X_train.columns for col in ordinales))
print('Columnas de X_train:', list(X_train.columns))


¿Están las numéricas? True
¿Están las nominales? True
¿Están las ordinales? True
Columnas de X_train: ['edad', 'antiguedad_meses', 'ingresos_anuales', 'gasto_mensual', 'num_compras', 'devoluciones', 'dispositivo', 'zona', 'nivel_fidelidad', 'satisfaccion']


**Responde:** ¿Qué variables tienen escalas más distintas? ¿Es obligatorio escalar para cualquier algoritmo? ¿Qué particularidad tiene `satisfaccion`?

**Respuesta:**  
- Las escalas más distintas son, por ejemplo, `ingresos_anuales` (miles) frente a `satisfaccion` (1-5) o `devoluciones` (números pequeños).  
- **No** es obligatorio para cualquier algoritmo: árboles y bosques suelen aguantar bien sin escalar; sí importa más en modelos por distancia o regresión logística.  
- `satisfaccion` es una escala 1-5 (ordinal), pero aquí la tratamos como **numérica** por simplificación.


## Parte 5 - Pipeline completo
### Ejercicio 13. Pipeline numérico
Construye un `Pipeline` que impute con mediana y después estandarice.

In [14]:
pipeline_numerico = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

pipeline_numerico


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account fo

**¿Por qué imputamos antes de escalar en este pipeline?**

**Respuesta:**  
Porque el escalado necesita valores numéricos válidos. Si hay `NaN`, primero los rellenamos y **después** calculamos media/desviación (o el Min-Max).


### Ejercicio 14. Pipeline nominal
Imputa con moda y después aplica One-Hot Encoding (`handle_unknown="ignore"`, `sparse_output=False`).

In [15]:
pipeline_nominal = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

pipeline_nominal


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('encoder', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'most_frequent'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto ac

### Ejercicio 15. Pipeline ordinal
Imputa con moda y codifica Bronce → Plata → Oro → Platino. Gestiona categorías desconocidas con `handle_unknown="use_encoded_value"` y `unknown_value=-1`.

In [16]:
pipeline_ordinal = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=[['Bronce', 'Plata', 'Oro', 'Platino']],
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

pipeline_ordinal


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('encoder', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'most_frequent'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto ac

### Ejercicio 16. ColumnTransformer
Combina los tres pipelines para aplicar cada transformación únicamente a sus columnas.

In [17]:
preprocesador = ColumnTransformer(transformers=[
    ('num', pipeline_numerico, numericas),
    ('nom', pipeline_nominal, nominales),
    ('ord', pipeline_ordinal, ordinales)
])

preprocesador


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('nom', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_n

### Ejercicio 17. Transformar entrenamiento y prueba
Ajusta el preprocesador **solo** con entrenamiento y aplícalo a prueba sin reajustarlo.

In [18]:
# fit_transform SOLO en entrenamiento
X_train_preparado = preprocesador.fit_transform(X_train)

# transform (sin reajustar) en prueba
X_test_preparado = preprocesador.transform(X_test)

print('Train preparado:', X_train_preparado.shape)
print('Test preparado:', X_test_preparado.shape)


Train preparado: (320, 16)
Test preparado: (80, 16)


**Responde:** ¿Por qué no usar `fit_transform()` en prueba? ¿Qué aprende `fit()`? ¿Qué ocurre si aparece una categoría nominal nueva en prueba?

**Respuesta:**  
- En prueba usamos solo `transform()` para no aprender parámetros con datos de test.  
- `fit()` aprende, por ejemplo, medianas, modas, medias/desviaciones y las categorías vistas en entrenamiento.  
- Si aparece una categoría nominal nueva en prueba, con `handle_unknown='ignore'` el One-Hot pone **ceros** en esas columnas (no rompe el pipeline).


## Parte 6 - Validación e informe
### Ejercicio 18. Verificar matrices
Comprueba tamaños, ausentes, tipos numéricos, correspondencia con las etiquetas y nombres de características.

In [19]:
print('Shape train:', X_train_preparado.shape)
print('Shape test:', X_test_preparado.shape)

print('NaN train:', np.isnan(X_train_preparado).sum())
print('NaN test:', np.isnan(X_test_preparado).sum())

print('Tipo train:', X_train_preparado.dtype)
print('Tipo test:', X_test_preparado.dtype)

print('Filas train vs y_train:', X_train_preparado.shape[0], len(y_train))
print('Filas test vs y_test:', X_test_preparado.shape[0], len(y_test))
print('¿Misma nº columnas?', X_train_preparado.shape[1] == X_test_preparado.shape[1])

print('Nombres de características:')
print(preprocesador.get_feature_names_out())


Shape train: (320, 16)
Shape test: (80, 16)
NaN train: 0
NaN test: 0
Tipo train: float64
Tipo test: float64
Filas train vs y_train: 320 320
Filas test vs y_test: 80 80
¿Misma nº columnas? True
Nombres de características:
['num__edad' 'num__antiguedad_meses' 'num__ingresos_anuales'
 'num__gasto_mensual' 'num__num_compras' 'num__devoluciones'
 'num__satisfaccion' 'nom__dispositivo_Móvil' 'nom__dispositivo_Ordenador'
 'nom__dispositivo_Tablet' 'nom__zona_Centro' 'nom__zona_Este'
 'nom__zona_Norte' 'nom__zona_Oeste' 'nom__zona_Sur'
 'ord__nivel_fidelidad']


**Responde:** ¿Por qué aumentaron las columnas? ¿Coinciden las características en ambos conjuntos? ¿Quedan ausentes? ¿Están alineadas las etiquetas?

**Respuesta:**  
- Aumentan las columnas por el **One-Hot** de `dispositivo` y `zona` (una columna por categoría).  
- Sí: train y test tienen las **mismas características** (mismo `preprocesador`).  
- No deberían quedar ausentes tras la imputación.  
- Sí están alineadas: el número de filas de cada matriz coincide con `y_train` / `y_test`.


### Ejercicio 19. Reconstruir DataFrames
Convierte las matrices preparadas en `df_train_preparado` y `df_test_preparado`. Usa `preprocesador.get_feature_names_out()` y conserva los índices de X_train/X_test.

In [20]:
columnas_finales = preprocesador.get_feature_names_out()

df_train_preparado = pd.DataFrame(
    X_train_preparado,
    columns=columnas_finales,
    index=X_train.index
)

df_test_preparado = pd.DataFrame(
    X_test_preparado,
    columns=columnas_finales,
    index=X_test.index
)

print(df_train_preparado.head())
print(df_test_preparado.head())


     num__edad  num__antiguedad_meses  num__ingresos_anuales  \
360  -0.068723              -1.095565               0.651045   
356  -0.677904               1.129874               1.422884   
138   0.845048              -1.540653               0.907575   
42   -1.408922               0.447406               0.476223   
216  -0.495150               1.456272              -0.493983   

     num__gasto_mensual  num__num_compras  num__devoluciones  \
360           -1.086877          0.277945          -0.185205   
356           -1.092939          0.277945          -1.069767   
138            0.537957         -0.457116          -1.069767   
42             0.303390          1.380537          -1.069767   
216            1.466735          0.645476          -1.069767   

     num__satisfaccion  nom__dispositivo_Móvil  nom__dispositivo_Ordenador  \
360          -0.254293                     0.0                         1.0   
356          -0.254293                     1.0                         0.0